# Tensor Operations: The Complete Reference

Reach for this when you need:
- Matrix multiplication variants (`mm`, `bmm`, `matmul`, `@`).
- Joining tensors along existing or new axes (`cat`, `stack`).
- Splitting tensors (`chunk`, `split`, `unbind`).
- Expressive multi-dimensional contractions with Einstein summation (`einsum`).
- Element-wise, reduction, and indexing-based ops used daily in DL code.

In [2]:
import torch
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


---
## 1. Matrix Multiplication

| Function | Shape | Use case |
| :--- | :--- | :--- |
| `torch.mm(A, B)` | `(m,k) × (k,n) → (m,n)` | 2-D only, strict |
| `torch.matmul(A, B)` / `A @ B` | Broadcasting-aware | General purpose |
| `torch.bmm(A, B)` | `(B,m,k) × (B,k,n) → (B,m,n)` | Batched 2-D, no broadcasting |

✅ **Use `@` / `matmul` in new code** — handles 2-D, batched, and broadcast cases uniformly.

In [4]:
# ── 2-D matrix multiply ────────────────────────────────────────────────────────
A = torch.randn(3, 4)
B = torch.randn(4, 5)

C1 = torch.mm(A, B)        # strict 2-D
C2 = A @ B                 # operator syntax (calls matmul)
C3 = torch.matmul(A, B)    # explicit

print('mm result:', C1.shape)   # (3, 5)
print("@ result:", C2.shape)
print("matmul result:", C3.shape)
print('All equal:', torch.allclose(C1, C2) and torch.allclose(C1, C3))

mm result: torch.Size([3, 5])
@ result: torch.Size([3, 5])
matmul result: torch.Size([3, 5])
All equal: True


In [5]:
# ── Batched matrix multiply ────────────────────────────────────────────────────
batch = 8
A_b = torch.randn(batch, 3, 4)
B_b = torch.randn(batch, 4, 5)

C_bmm = torch.bmm(A_b, B_b)       # (8, 3, 5)
C_mat = A_b @ B_b                  # identical result via matmul

print('bmm result:', C_bmm.shape)
print('@ result:', C_mat.shape)
print('Equal to @:', torch.allclose(C_bmm, C_mat))

bmm result: torch.Size([8, 3, 5])
@ result: torch.Size([8, 3, 5])
Equal to @: True


In [6]:
# ── Broadcasting matmul (new feature-rich behavior) ────────────────────────────
# matmul broadcasts over all leading dimensions, like NumPy
X = torch.randn(2, 3, 4, 5)   # 4-D tensor
W = torch.randn(5, 6)          # 2-D weight matrix

out = X @ W     # broadcasts: (2, 3, 4, 5) x (5, 6) → (2, 3, 4, 6)
print('Broadcast matmul:', out.shape)

Broadcast matmul: torch.Size([2, 3, 4, 6])


---
## 2. Vector Products

| Function | Result | Description |
| :--- | :--- | :--- |
| `torch.dot(a, b)` | scalar | 1-D inner product |
| `torch.inner(A, B)` | tensor | Last-dim contraction, multi-dim |
| `torch.outer(a, b)` | `(n, m)` | Rank-1 outer product |

In [9]:
a = torch.tensor([1., 2., 3.]) # (1, 3)
b = torch.tensor([4., 5., 6.]) # (1, 3)

dot    = torch.dot(a, b)        # 1*4 + 2*5 + 3*6 = 32.0, does (axb)*(cxb).T
inner = torch.inner(a, b)       # 
outer  = torch.outer(a, b)      # shape (3, 3)

print('Dot product:\n', dot.item())
print('\nInner product:\n', inner)
print('\nOuter product:\n', outer)

Dot product:
 32.0

Inner product:
 tensor(32.)

Outer product:
 tensor([[ 4.,  5.,  6.],
        [ 8., 10., 12.],
        [12., 15., 18.]])


In [10]:
# ── Cross product (3-D vectors only) ───────────────────────────────────────────
v1 = torch.tensor([1., 0., 0.])
v2 = torch.tensor([0., 1., 0.])

cross = torch.linalg.cross(v1, v2)   # [0, 0, 1] — x cross y = z
print('Cross product:', cross)

Cross product: tensor([0., 0., 1.])


---
## 3. Joining Tensors: `cat` vs `stack`

| Function | Effect on rank | When to use |
| :--- | :--- | :--- |
| `torch.cat(tensors, dim)` | **Same rank** — concatenates along existing dim | Combine batch of sequences, grow sequence length |
| `torch.stack(tensors, dim)` | **Rank + 1** — inserts a new dimension and stacks | Pack individual samples into a batch |

In [12]:
# ── torch.cat ─────────────────────────────────────────────────────────────────
t1 = torch.ones(2, 3)
t2 = torch.zeros(2, 3)

cat_dim0 = torch.cat([t1, t2], dim=0)   # (4, 3) — glue rows
cat_dim1 = torch.cat([t1, t2], dim=1)   # (2, 6) — glue columns

print('cat dim=0:\n', cat_dim0)
print('\ncat dim=1:\n', cat_dim1)

cat dim=0:
 tensor([[1., 1., 1.],
        [1., 1., 1.],
        [0., 0., 0.],
        [0., 0., 0.]])

cat dim=1:
 tensor([[1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.]])


In [13]:
# ── torch.stack ───────────────────────────────────────────────────────────────
frames = [torch.randn(3, 224, 224) for _ in range(4)]  # 4 image tensors

batch       = torch.stack(frames, dim=0)  # (4, 3, 224, 224) — classic batch stacking
time_series = torch.stack(frames, dim=1)  # (3, 4, 224, 224) — stack along time axis

print('stack dim=0 (batch):', batch.shape)
print('stack dim=1 (time):', time_series.shape)

stack dim=0 (batch): torch.Size([4, 3, 224, 224])
stack dim=1 (time): torch.Size([3, 4, 224, 224])


---
## 4. Splitting Tensors: `chunk`, `split`, `unbind`

| Function | Description |
| :--- | :--- |
| `torch.chunk(t, n, dim)` | Split into ≤ n equal-ish pieces along `dim` |
| `torch.split(t, size, dim)` | Split by explicit size (or list of sizes) |
| `torch.unbind(t, dim)` | Remove a dimension, return tuple of slices |

In [14]:
x = torch.arange(12, dtype=torch.float).reshape(4, 3)
print('Original:', x.shape, '\n', x)

# chunk: split into 2 equal halves along dim=0
pieces = torch.chunk(x, 2, dim=0)    # tuple of two (2, 3) tensors
print('\nchunk:', [p.shape for p in pieces])

# split: split into sizes [1, 3] along dim=0
parts = torch.split(x, [1, 3], dim=0) # (1,3) and (3,3)
print('split:', [p.shape for p in parts])

# unbind: remove dim=0, returns 4 tensors of shape (3,)
rows = torch.unbind(x, dim=0)
print('unbind:', [r.shape for r in rows])

Original: torch.Size([4, 3]) 
 tensor([[ 0.,  1.,  2.],
        [ 3.,  4.,  5.],
        [ 6.,  7.,  8.],
        [ 9., 10., 11.]])

chunk: [torch.Size([2, 3]), torch.Size([2, 3])]
split: [torch.Size([1, 3]), torch.Size([3, 3])]
unbind: [torch.Size([3]), torch.Size([3]), torch.Size([3]), torch.Size([3])]


---
## 5. Einstein Summation: `torch.einsum`

**`einsum` lets you write any tensor contraction in a single readable string.**

The string `'ij,jk->ik'` means:
- `i, j` index the first operand
- `j, k` index the second operand  
- `j` is repeated (summed over) → `i, k` survive in the output

| Expression | Operation | Equivalent |
| :--- | :--- | :--- |
| `'i,i->'` | Dot product | `torch.dot` |
| `'ij,jk->ik'` | Matrix multiply | `A @ B` |
| `'bij,bjk->bik'` | Batched matmul | `torch.bmm` |
| `'i,j->ij'` | Outer product | `torch.outer` |
| `'ii->'` | Trace | `torch.trace` |
| `'ij->ji'` | Transpose | `.T` |
| `'bqk,bvk->bqv'` | Attention scores | Used in self-attention |

In [19]:
# ── Basic einsum examples ──────────────────────────────────────────────────────
A = torch.randn(3, 4)
B = torch.randn(4, 5)
v = torch.randn(4)

# Matrix multiply
mm_result = torch.einsum('ij,jk->ik', A, B)
print('matmul via einsum:', mm_result.shape)     # (3, 5)

# Matrix-vector product
mv_result = torch.einsum('ij,j->i', A, v)
print('mat-vec via einsum:', mv_result.shape)    # (3,)

# Transpose
At = torch.einsum('ij->ji', A)
print('transpose via einsum:', At.shape)         # (4, 3)

# Trace
sq = torch.randn(4, 4)
trace_e = torch.einsum('ii->', sq)
trace_t = torch.trace(sq)
print(f'Trace: einsum={trace_e.item():.4f} | torch.trace={trace_t.item():.4f}')

matmul via einsum: torch.Size([3, 5])
mat-vec via einsum: torch.Size([3])
transpose via einsum: torch.Size([4, 3])
Trace: einsum=-3.2382 | torch.trace=-3.2382


In [36]:
# ── Batched einsum — self-attention style ──────────────────────────────────────
# Q: (batch, heads, seq, dim_k)
# K: (batch, heads, seq, dim_k)
# Output: (batch, heads, seq, seq)  — the attention matrix

B_sz, H, S, D = 2, 4, 8, 16
Q = torch.randn(B_sz, H, S, D)
K = torch.randn(B_sz, H, S, D)

# Standard: Q @ K^T  per head and batch
attn_scores = torch.einsum('bhqd,bhkd->bhqk', Q, K)
print('Attention scores shape:', attn_scores.shape)  # (2, 4, 8, 8)

Attention scores shape: torch.Size([2, 4, 8, 8])


In [37]:
# ── Outer product and element-wise contractions ────────────────────────────────
a = torch.randn(3)
b = torch.randn(5)

outer_e = torch.einsum('i,j->ij', a, b)   # (3, 5)
hadamard = torch.einsum('ij,ij->ij', torch.randn(3,5), torch.randn(3,5))  # element-wise

print('Outer product:', outer_e.shape)
print('Hadamard product:', hadamard.shape)

Outer product: torch.Size([3, 5])
Hadamard product: torch.Size([3, 5])


---
## 6. Advanced Indexing: `gather`, `scatter`, `index_select`

| Function | Description | Common use |
| :--- | :--- | :--- |
| `torch.gather(src, dim, idx)` | Collect values by index | Picking predicted class logits |
| `torch.scatter_(dst, dim, idx, src)` | Write values by index (in-place) | One-hot encoding, label smoothing |
| `torch.index_select(t, dim, idx)` | Subset rows/cols by 1D index tensor | Vocabulary lookup |

In [ ]:
# ── gather: pick specific columns per row ──────────────────────────────────────
# Classic use: select the logit for the correct class from a (batch, classes) tensor
logits = torch.tensor([[0.1, 0.9, 0.3],
                        [0.8, 0.2, 0.6],
                        [0.4, 0.5, 0.7]])
labels = torch.tensor([[1], [0], [2]])   # correct class per sample

selected = logits.gather(dim=1, index=labels)  # (3, 1)
print('Selected logits:\n', selected)

In [ ]:
# ── scatter_: build one-hot matrix from class indices ─────────────────────────
labels_1d = torch.tensor([2, 0, 1])
num_classes = 4

one_hot = torch.zeros(3, num_classes)
one_hot.scatter_(dim=1, index=labels_1d.unsqueeze(1), value=1.0)
print('One-hot:\n', one_hot)

# Or using the functional API:
one_hot_f = F.one_hot(labels_1d, num_classes=num_classes).float()
print('F.one_hot:\n', one_hot_f)

In [ ]:
# ── index_select ───────────────────────────────────────────────────────────────
matrix = torch.arange(20).reshape(4, 5).float()
idx    = torch.tensor([0, 2])   # select rows 0 and 2

rows = torch.index_select(matrix, dim=0, index=idx)
print('index_select (rows 0,2):\n', rows)

---
## 7. Reduction Operations

Reductions collapse one or all dimensions to a scalar.

| Function | Description |
| :--- | :--- |
| `sum`, `mean`, `std`, `var` | Standard statistics |
| `min`, `max`, `argmin`, `argmax` | Extrema and their indices |
| `norm` | L2 norm (or Lp for any p) |
| `prod` | Product of elements |
| `cumsum`, `cumprod` | Cumulative reductions |

In [ ]:
x = torch.arange(1., 13.).reshape(3, 4)
print('x:\n', x)

print('sum all:     ', x.sum().item())
print('sum dim=1:   ', x.sum(dim=1))         # sum across columns → shape (3,)
print('mean dim=0:  ', x.mean(dim=0))         # mean across rows → shape (4,)
print('L2 norm:     ', x.norm(p=2).item())
print('argmax dim=1:', x.argmax(dim=1))       # index of biggest value per row

# keepdim=True preserves the reduced dimension (useful for broadcasting)
row_sums = x.sum(dim=1, keepdim=True)   # shape (3, 1)
normed   = x / row_sums                 # broadcast: each row divided by its sum
print('Row-normalized (row sums should be 1):', normed.sum(dim=1))

---
## 8. Sorting and Top-K

| Function | Description |
| :--- | :--- |
| `torch.sort(t, dim)` | Returns sorted values + original indices |
| `torch.topk(t, k, dim)` | Returns k largest values + their indices |
| `torch.argsort(t, dim)` | Indices that would sort the tensor |

In [ ]:
scores = torch.tensor([0.15, 0.80, 0.40, 0.95, 0.25])

sorted_vals, sorted_idx = torch.sort(scores, descending=True)
print('Sorted values:', sorted_vals)
print('Original indices:', sorted_idx)

top3_vals, top3_idx = torch.topk(scores, k=3)
print('Top-3 values:', top3_vals)
print('Top-3 indices:', top3_idx)

---
## 9. Unfolding Windows: `unfold`

`unfold` is the sliding-window operator: extract all overlapping windows of a given `size` spaced `step` apart.

In [ ]:
# ── 1-D sliding windows ────────────────────────────────────────────────────────
signal = torch.arange(10, dtype=torch.float)   # [0, 1, 2, ..., 9]

windows = signal.unfold(dimension=0, size=3, step=1)
print('Signal:', signal)
print('Unfold (size=3, step=1):\n', windows)
print('Shape:', windows.shape)  # (8, 3)

---
## 10. Tensor Combination Utilities

`torch.where`, `torch.masked_fill`, `torch.nan_to_num`

In [ ]:
# ── torch.where: conditional element selection ─────────────────────────────────
a = torch.tensor([1., -2., 3., -4.])
b = torch.zeros_like(a)

relu_like = torch.where(a > 0, a, b)   # positive values kept, negatives → 0
print('ReLU via where:', relu_like)

In [ ]:
# ── masked_fill: write a fill value wherever mask is True ─────────────────────
# Classic use: causal attention mask in Transformers
seq_len = 5
mask = torch.tril(torch.ones(seq_len, seq_len)).bool()  # lower triangular
causal_mask = torch.zeros(seq_len, seq_len)
causal_mask = causal_mask.masked_fill(~mask, float('-inf'))   # fill UPPER with -inf
print('Causal attention mask (pre-softmax):\n', causal_mask)

In [ ]:
# ── nan_to_num: replace NaN/Inf safely ────────────────────────────────────────
t = torch.tensor([1., float('nan'), float('inf'), -float('inf')])
clean = torch.nan_to_num(t, nan=0.0, posinf=1e9, neginf=-1e9)
print('Cleaned:', clean)

---
## 11. Linear Algebra Cheat Sheet (`torch.linalg`)

| Function | Description |
| :--- | :--- |
| `linalg.norm` | Matrix or vector norms |
| `linalg.inv` | Matrix inverse |
| `linalg.det` | Determinant |
| `linalg.eig` | Eigenvalues and eigenvectors |
| `linalg.svd` | Singular value decomposition |
| `linalg.qr` | QR decomposition |
| `linalg.solve` | Solve Ax=b |

In [ ]:
# ── SVD — used inside PCA, LoRA, and low-rank approximations ──────────────────
A = torch.randn(4, 6)

U, S, Vh = torch.linalg.svd(A, full_matrices=False)
print(f'U: {U.shape}, S: {S.shape}, Vh: {Vh.shape}')

# Reconstruct from singular values (low-rank approximation with top-r singular values)
r = 2
A_approx = U[:, :r] @ torch.diag(S[:r]) @ Vh[:r, :]
print(f'Original rank-4 approx via rank-{r}: {A_approx.shape}')
print(f'Reconstruction error: {(A - A_approx).norm().item():.4f}')

In [ ]:
# ── Solve linear system Ax = b ─────────────────────────────────────────────────
A_sq = torch.randn(3, 3)
b    = torch.randn(3)

x = torch.linalg.solve(A_sq, b)   # A @ x ≈ b
residual = (A_sq @ x - b).norm()
print(f'Residual ||Ax - b||: {residual.item():.2e}')  # should be ~0

---
## 12. `repeat` and `expand` vs `tile`

| Function | Memory | Modifies shape | |
| :--- | :--- | :--- | :--- |
| `.expand(*sizes)` | **View** (no copy) | Grow size-1 dims only | Fastest |
| `.repeat(*reps)` | **Copies** data | Any dim | Flexible |
| `torch.tile(t, reps)` | **Copies** data | Like np.tile | NumPy-compatible |

In [ ]:
x = torch.tensor([[1., 2., 3.]])   # shape (1, 3)

# expand: broadcast-style, no memory copy — rows don't physically exist
x_expanded = x.expand(4, 3)       # (4, 3) — shares storage with x
print('expand:', x_expanded.shape)

# repeat: actually copies the data
x_repeated = x.repeat(4, 2)       # (4, 6) — 4 repeats in dim=0, 2 in dim=1
print('repeat:', x_repeated.shape)

# tile: NumPy-style
x_tiled = torch.tile(x, (3, 2))   # (3, 6)
print('tile:  ', x_tiled.shape)

---
## 13. Flatten and Reshape Utilities

| Function | Description |
| :--- | :--- |
| `flatten(start_dim, end_dim)` | Collapse a range of dims into one |
| `reshape` | Total reshape (safe version of view) |
| `squeeze` / `unsqueeze` | Remove / add size-1 dims |
| `contiguous()` | Copy data into a contiguous block (needed before view) |

In [ ]:
# Feature map from a CNN: (batch, channels, H, W)
feature_map = torch.randn(8, 16, 14, 14)

# Flatten spatial dims only (keep batch and channel)
flat_spatial = feature_map.flatten(start_dim=2)    # (8, 16, 196)
print('Flatten spatial:', flat_spatial.shape)

# Flatten everything except batch
flat_all = feature_map.flatten(start_dim=1)        # (8, 3136)
print('Flatten all but batch:', flat_all.shape)

# Contiguous check
transposed = feature_map.transpose(1, 2)           # non-contiguous
print('Is contiguous after transpose:', transposed.is_contiguous())
print('After .contiguous():', transposed.contiguous().is_contiguous())

---
### Key Takeaways

| Topic | Rule of Thumb |
| :--- | :--- |
| Matmul | Use `@` for clarity; use `bmm` when you want explicit no-broadcast batched multiply |
| `cat` vs `stack` | `cat` merges, `stack` adds a dimension |
| `einsum` | The most flexible tensor op — learn `'ij,jk->ik'` and `'bhqd,bhkd->bhqk'` patterns |
| `gather` | Essential for NLL loss, picking top-k logits, etc. |
| `expand` | Prefer over `repeat` when possible — it's free (no copy) |
| `unfold` | Sliding windows for 1-D and 2-D data, manual conv-like ops |
| `linalg.svd` | Foundation of LoRA, PCA, and stable numerical computation |